# Nível 1
**Passo 1 — Criar a conexão**

In [1]:
from sqlalchemy import create_engine

engine = create_engine("sqlite:///sistema_rh.db")

print("Conexão criada com sucesso!")

Conexão criada com sucesso!


**Passo 2 — Criar a tabela funcionarios**

In [2]:
from sqlalchemy import create_engine, text

engine = create_engine("sqlite:///sistema_rh.db")

with engine.begin() as conn:
    conn.execute(text("""
        CREATE TABLE IF NOT EXISTS funcionarios (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            nome TEXT NOT NULL,
            cargo TEXT NOT NULL,
            salario REAL NOT NULL
        )
    """))

print("Tabela funcionarios criada com sucesso!")

Tabela funcionarios criada com sucesso!


**Passo 3 — Inserção segura**

In [3]:
from sqlalchemy import create_engine, text

engine = create_engine("sqlite:///sistema_rh.db")

nome = "João"
cargo = "Desenvolvedor Júnior"
salario = 3500.00

with engine.begin() as conn:
    conn.execute(
        text("""
            INSERT INTO funcionarios (nome, cargo, salario)
            VALUES (:nome, :cargo, :salario)
        """),
        {
            "nome": nome,
            "cargo": cargo,
            "salario": salario
        }
    )

print("Funcionário inserido com sucesso!")

Funcionário inserido com sucesso!


**Passo 4 — Consultar com Pandas**

In [4]:
from sqlalchemy import create_engine
import pandas as pd

engine = create_engine("sqlite:///sistema_rh.db")

df = pd.read_sql_query(
    "SELECT * FROM funcionarios",
    engine
)

print(df)

   id  nome                 cargo  salario
0   1  João  Desenvolvedor Júnior   3500.0


# Nível 2
**Passo 1 — Criar tabela projetos**

In [5]:
from sqlalchemy import (
    create_engine,
    MetaData,
    Table,
    Column,
    Integer,
    String
)

engine = create_engine("sqlite:///sistema_rh.db")

metadata = MetaData()

projetos = Table(
    "projetos",
    metadata,
    Column("id", Integer, primary_key=True),
    Column("nome", String, nullable=False),
    Column("responsavel", String, nullable=False)
)

metadata.create_all(engine)

print("Tabela projetos criada com sucesso!")

Tabela projetos criada com sucesso!


**Passo 2 — Inserção em lote**

In [6]:
from sqlalchemy import create_engine, MetaData, Table, insert

engine = create_engine("sqlite:///sistema_rh.db")

metadata = MetaData()

projetos = Table(
    "projetos",
    metadata,
    autoload_with=engine
)

lista_de_projetos = [
    {
        "nome": "Sistema de Vendas",
        "responsavel": "Ana"
    },
    {
        "nome": "Sistema de Estoque",
        "responsavel": "Carlos"
    },
    {
        "nome": "Sistema Financeiro",
        "responsavel": "Mariana"
    }
]

with engine.begin() as conn:
    conn.execute(
        insert(projetos),
        lista_de_projetos
    )

print("Projetos inseridos com sucesso!")

Projetos inseridos com sucesso!


**Passo 3 — Reajustar salário**

In [7]:
from sqlalchemy import (
    create_engine,
    MetaData,
    Table,
    update
)

engine = create_engine("sqlite:///sistema_rh.db")

metadata = MetaData()

funcionarios = Table(
    "funcionarios",
    metadata,
    autoload_with=engine
)

comando = (
    update(funcionarios)
    .where(funcionarios.c.cargo == "Desenvolvedor Júnior")
    .values(
        salario=funcionarios.c.salario * 1.10
    )
)

with engine.begin() as conn:
    resultado = conn.execute(comando)

print("Reajuste realizado com sucesso!")
print("Funcionários atualizados:", resultado.rowcount)

Reajuste realizado com sucesso!
Funcionários atualizados: 1


**Passo 4 — Relatório salarial**

In [8]:
from sqlalchemy import (
    create_engine,
    MetaData,
    Table,
    select,
    func
)

engine = create_engine("sqlite:///sistema_rh.db")

metadata = MetaData()

funcionarios = Table(
    "funcionarios",
    metadata,
    autoload_with=engine
)

consulta = (
    select(
        funcionarios.c.cargo,
        func.avg(funcionarios.c.salario).label("media_salarial")
    )
    .group_by(funcionarios.c.cargo)
)

with engine.connect() as conn:
    resultado = conn.execute(consulta)

    for linha in resultado:
        print(
            "Cargo:", linha.cargo,
            "| Média salarial:", round(linha.media_salarial, 2)
        )

Cargo: Desenvolvedor Júnior | Média salarial: 3850.0


# Nível 3

**Passo 1 — Classes Departamento e FuncionarioORM**

In [10]:
from sqlalchemy import create_engine, String, Float
from sqlalchemy.orm import (
    declarative_base,
    Mapped,
    mapped_column
)

engine = create_engine("sqlite:///sistema_rh.db")

Base = declarative_base()


class Departamento(Base):
    __tablename__ = "departamentos_orm"

    id: Mapped[int] = mapped_column(primary_key=True)
    nome: Mapped[str] = mapped_column(String, nullable=False)


class FuncionarioORM(Base):
    __tablename__ = "funcionarios_orm"

    id: Mapped[int] = mapped_column(primary_key=True)
    nome: Mapped[str] = mapped_column(String, nullable=False)
    cargo: Mapped[str] = mapped_column(String, nullable=False)
    salario: Mapped[float] = mapped_column(Float, nullable=False)


Base.metadata.create_all(engine)

print("Tabelas ORM criadas com sucesso!")

Tabelas ORM criadas com sucesso!


**Passo 2 — Relacionamento**

In [15]:
from sqlalchemy import (
    create_engine,
    String,
    Float,
    ForeignKey
)

from sqlalchemy.orm import (
    declarative_base,
    Mapped,
    mapped_column,
    relationship
)

engine = create_engine("sqlite:///sistema_rh.db")

Base = declarative_base()


class Departamento(Base):
    __tablename__ = "departamentos_orm"

    id: Mapped[int] = mapped_column(primary_key=True)
    nome: Mapped[str] = mapped_column(String, nullable=False)

    funcionarios: Mapped[list["FuncionarioORM"]] = relationship(
        back_populates="departamento"
    )


class FuncionarioORM(Base):
    __tablename__ = "funcionarios_orm"

    id: Mapped[int] = mapped_column(primary_key=True)
    nome: Mapped[str] = mapped_column(String, nullable=False)
    cargo: Mapped[str] = mapped_column(String, nullable=False)
    salario: Mapped[float] = mapped_column(Float, nullable=False)

    departamento_id: Mapped[int] = mapped_column(
        ForeignKey("departamentos_orm.id")
    )

    departamento: Mapped["Departamento"] = relationship(
        back_populates="funcionarios"
    )


Base.metadata.create_all(engine)

print("Relacionamento criado com sucesso!")

Relacionamento criado com sucesso!


**Passo 3 — Session + Inserção**

In [16]:
from sqlalchemy.orm import sessionmaker

Session = sessionmaker(bind=engine)
sessao = Session()

departamento_ti = Departamento(nome="TI")

funcionario1 = FuncionarioORM(
    nome="Carlos",
    cargo="Desenvolvedor",
    salario=5000.00
)

funcionario2 = FuncionarioORM(
    nome="Mariana",
    cargo="Analista de Sistemas",
    salario=4500.00
)

departamento_ti.funcionarios.append(funcionario1)
departamento_ti.funcionarios.append(funcionario2)

sessao.add(departamento_ti)
sessao.commit()

print("Departamento e funcionários cadastrados com sucesso!")

Departamento e funcionários cadastrados com sucesso!


**Passo 4 — Consultar funcionários de TI**

In [17]:
from sqlalchemy import select

consulta = (
    select(FuncionarioORM)
    .join(FuncionarioORM.departamento)
    .where(Departamento.nome == "TI")
)

resultado = sessao.execute(
    consulta
).scalars().all()

print("Funcionários do departamento de TI:")

for funcionario in resultado:
    print(
        "Nome:", funcionario.nome,
        "| Cargo:", funcionario.cargo,
        "| Salário:", funcionario.salario
    )

sessao.close()

Funcionários do departamento de TI:
Nome: Carlos | Cargo: Desenvolvedor | Salário: 5000.0
Nome: Mariana | Cargo: Analista de Sistemas | Salário: 4500.0
